# PKG deposit balances by customer (from COD deposit_master)

Source: `codhd01p_tm.deposit_master`, daily account balances on business days, partitioned by `src_load_date` (balance date) and `data_source` (HOGAN = DDAs, TDA = CDs, AGILETICS_SWEEP = sweeps). Daily since mid-2016.

Outputs in `bdahd01p_dlcdi1_cdi_tm`, partitioned by `month` = calendar month of the balances (joins to PKG tables and `pkg_tm_cust_month` on customer_pwr_id + month):
- `pkg_dep_cust_month`: one row per power id per month, for power ids in `pkg_cust_dim` (people and organisations). Daily-balance statistics (month-end, mean, min, max, volatility, largest drop from a peak, days overdrawn), balances by deposit family, account events (new, closed, gone, turned dormant), earnings credit and interest.
- `pkg_dep_cust_daily`: one row per power id per business day with accounts, organisations only (party_type O): balance by family, accounts, overdrawn accounts.

Rules
- Account = data_source + acct_full_acct_id; CD certificates on the same account are summed; exact repeats of a row (same account, certificate, date) keep the latest load.
- Families: NIBDDA / IBDDA = business DDA, Retail, MMDA, CD, sweep on balance sheet, sweep off balance sheet (investment funds, kept apart: **dep_bal excludes them**), other.
- Business days = union of the sources' load dates in the month. A source missing a day (or with a partial load, under half its usual rows) carries its last balance forward up to 7 days; the run log counts carried days.
- Monthly statistics run over every business day of the month; a customer with no account that day counts as 0.
- Account events compare the month with the previous month-end: new = not there at the previous month-end; closed = status CLOSED at month-end and not before; gone = there at the previous month-end, absent all month; dormant = status DORMANT or INACTIVE at month-end and not before.
- A month counts as done only when both tables carry a marker with the current configuration signature; months are independent (the previous month-end is read from the source).

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("pkg_dep_cust")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
DB           = "bdahd01p_dlcdi1_cdi_tm"
T_MONTH      = "pkg_dep_cust_month"     # one row per power id per month (PKG customers)
T_DAILY      = "pkg_dep_cust_daily"     # one row per power id per business day (organisations only)
DEP_MASTER   = "codhd01p_tm.deposit_master"
PKG_DIM      = f"{DB}.pkg_cust_dim"
SOURCES      = ["HOGAN", "TDA", "AGILETICS_SWEEP"]   # DDAs, CDs, sweeps (data_source partition values)
START_MONTH  = "2023-01"
END_MONTH    = "latest"     # or "YYYY-MM"; latest = last month every source has finished
REBUILD_FROM = None         # "YYYY-MM": rebuild this month and every later one
WRITE_DAILY  = True
LOW_LOAD_SHARE = 0.5        # a load date with < this share of the source's median rows that month is treated as missing
CARRY_DAYS   = 7            # a source missing a business day carries its last balance forward up to this many days
HDFS_REPLICATION = 2       # copies per written file (quota counts size x copies); cluster default is 3
MIN_FREE_TB  = 0.2          # stop before a month when the cdi_tm quota has less than this free (TB, counted with copies)
VERSION      = "dep_cust_v1"
OUT_DIR      = "../metrics/dep_cust"
os.makedirs(OUT_DIR, exist_ok=True)
spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")   # month 'YYYY-MM' stays text
spark.sparkContext._jsc.hadoopConfiguration().set("dfs.replication", str(HDFS_REPLICATION))
spark.conf.set("dfs.replication", str(HDFS_REPLICATION))   # copied into each write job's Hadoop configuration

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("Source", DEP_MASTER), ("Output", f"{DB}.{T_MONTH}" + (f", {DB}.{T_DAILY}" if WRITE_DAILY else "")),
                        ("Months", f"{START_MONTH} to {END_MONTH}"), ("REBUILD_FROM", REBUILD_FROM),
                        ("HDFS_REPLICATION", HDFS_REPLICATION), ("MIN_FREE_TB", MIN_FREE_TB),
                        ("VERSION", VERSION), ("OUT_DIR", os.path.abspath(OUT_DIR))], columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Inputs, scope, month plan and resume status
# ============================================================
import json, hashlib, uuid, time
from urllib.parse import unquote

def ym_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"

def clean_pwr(c):
    """Power id as 10-digit text; '-1', blanks, zeros and non-digits -> NULL."""
    s = F.trim(F.col(c).cast("string"))
    s = F.when(s.rlike(r"^[0-9]{1,10}$"), F.lpad(s, 10, "0"))
    return F.when(s != F.lit("0000000000"), s)

# ---- Hadoop file helpers (markers) ----
_jvm, _hconf = spark._jvm, spark._jsc.hadoopConfiguration()
def _fp(p):
    P = _jvm.org.apache.hadoop.fs.Path(p)
    return P.getFileSystem(_hconf), P
def fs_write(p, text):
    fs, P = _fp(p)
    out = fs.create(P, True)
    out.write(bytearray(text.encode("utf-8")))
    out.close()
def fs_read(p):
    fs, P = _fp(p)
    if not fs.exists(P):
        return None
    rd = _jvm.java.io.BufferedReader(_jvm.java.io.InputStreamReader(fs.open(P), "UTF-8"))
    lines, line = [], rd.readLine()
    while line is not None:
        lines.append(line)
        line = rd.readLine()
    rd.close()
    return "\n".join(lines)
def fs_delete(p):
    fs, P = _fp(p)
    if fs.exists(P):
        fs.delete(P, True)

def quota(path):
    """(folder holding the space quota, quota bytes, consumed bytes) for path or its nearest parent with a quota;
    (None, None, consumed) when no folder up the tree has one. Bytes are counted with copies."""
    fs, P = _fp(path)
    q = P
    while q is not None:
        try:
            cs = fs.getContentSummary(q)
            if cs.getSpaceQuota() >= 0:
                return str(q), cs.getSpaceQuota(), cs.getSpaceConsumed()
        except Exception:
            pass
        q = q.getParent()
    return None, None, fs.getContentSummary(P).getSpaceConsumed()

def space_used(path):
    fs, P = _fp(path)
    return fs.getContentSummary(P).getSpaceConsumed() if fs.exists(P) else 0

def set_replication(path, n):
    """Set n copies on every file under path (covers writers that ignore dfs.replication); returns files changed."""
    fs, P = _fp(path)
    if not fs.exists(P):
        return 0
    it, k = fs.listFiles(P, True), 0
    while it.hasNext():
        st = it.next()
        if st.getReplication() > n and fs.setReplication(st.getPath(), n):
            k += 1
    return k

# ---- source columns (names from the 2026-11-10 profile) ----
DM = spark.table(DEP_MASTER)
need = ["acct_full_acct_id", "certificate_number", "cust_pwr_id", "deposit_family", "balance", "acct_status_desc",
        "earnings_credit_rate", "curr_int_rate", "interest_paid_mtd", "cod_hdfs_load_ts", "src_load_date", "data_source"]
miss = [c for c in need if c not in DM.columns]
if miss:
    raise RuntimeError(f"{DEP_MASTER} lacks columns {miss}")
pkd = spark.table(PKG_DIM)
miss = [c for c in ("customer_pwr_id", "party_type") if c not in pkd.columns]
if miss:
    raise RuntimeError(f"{PKG_DIM} lacks columns {miss}")

# ---- scope: PKG power ids; organisation when any of its mdm_ids is party_type O ----
SCOPE = (pkd.select(clean_pwr("customer_pwr_id").alias("customer_pwr_id"),
                    (F.upper(F.trim(F.col("party_type").cast("string"))) == "O").cast("int").alias("o"))
            .filter(F.col("customer_pwr_id").isNotNull())
            .groupBy("customer_pwr_id").agg(F.max("o").alias("is_org"))
            .persist(StorageLevel.MEMORY_AND_DISK))
sc = SCOPE.agg(F.count(F.lit(1)).alias("n"), F.sum("is_org").alias("org")).collect()[0]
if not sc["n"]:
    raise RuntimeError(f"No power ids in {PKG_DIM}")

# ---- partitions (metadata only) ----
pp = spark.sql(f"SHOW PARTITIONS {DEP_MASTER}").toPandas().iloc[:, 0].astype(str)
PT = pd.DataFrame([dict(kv.split("=", 1) for kv in v.split("/")) for v in pp]).apply(lambda s: s.map(unquote))
PT["d"] = pd.to_datetime(PT["src_load_date"], errors="coerce")
PT = PT[PT["d"].notna() & PT["data_source"].isin(SOURCES)]
SRC_DATES = {s: sorted(PT.loc[PT["data_source"] == s, "src_load_date"].unique().tolist()) for s in SOURCES}
empty = [s for s in SOURCES if not SRC_DATES[s]]
if empty:
    raise RuntimeError(f"No partitions for data_source {empty} in {DEP_MASTER}")

def ym_of(dstr):
    return dstr[:7]

# last finished month: every source already has a date in a later month
last_ok = ym_add(min(ym_of(SRC_DATES[s][-1]) for s in SOURCES), -1)
END = last_ok if END_MONTH == "latest" else END_MONTH
MONTHS, m = [], START_MONTH
while m <= END:
    MONTHS.append(m)
    m = ym_add(m, 1)
gaps = [f"{m}: no {s} dates" for m in MONTHS for s in SOURCES if not any(ym_of(d) == m for d in SRC_DATES[s])]
gaps += [f"{ym_add(MONTHS[0], -1)}: no {s} dates (needed for the first month's previous month-end)"
         for s in SOURCES if MONTHS and not any(ym_of(d) == ym_add(MONTHS[0], -1) for d in SRC_DATES[s])]
if not MONTHS or gaps:
    raise RuntimeError("Month plan incomplete: " + ("; ".join(gaps[:20]) if gaps else "no months"))

# ---- output locations, signature, resume status ----
_d = spark.sql(f"DESCRIBE DATABASE EXTENDED {DB}").toPandas()
LOC = str(_d.loc[_d.iloc[:, 0].astype(str).str.strip().str.lower() == "location"].iloc[0, 1]).rstrip("/")
PATH_M, PATH_D = f"{LOC}/{T_MONTH}", f"{LOC}/{T_DAILY}"
SIG = hashlib.sha1(json.dumps({"v": VERSION, "src": DEP_MASTER, "scope": PKG_DIM, "sources": SOURCES,
                               "low": LOW_LOAD_SHARE, "carry": CARRY_DAYS, "daily": WRITE_DAILY},
                              sort_keys=True).encode()).hexdigest()[:12]

def marker_ok(m):
    a = fs_read(f"{PATH_M}/month={m}/_PKG_MARKER.json")
    if not a:
        return False
    if WRITE_DAILY and fs_read(f"{PATH_D}/month={m}/_PKG_MARKER.json") != a:
        return False
    try:
        return json.loads(a).get("sig") == SIG
    except ValueError:
        return False

def plan_months(use_rebuild=True):
    return [m for m in MONTHS if not marker_ok(m) or (use_rebuild and REBUILD_FROM and m >= REBUILD_FROM)]

QF, QB, QU = quota(LOC)
TB = 1e12
QTXT = (f"{QU / TB:.2f} of {QB / TB:.2f} TB used, {(QB - QU) / TB:.2f} TB free (quota on {QF})" if QB is not None
        else f"no space quota found up the tree; folder uses {QU / TB:.2f} TB")
if QB is not None and QB - QU < MIN_FREE_TB * TB:
    show([f"<h3 class='warn'>cdi_tm quota: {esc(QTXT)}</h3><p>Less than {MIN_FREE_TB} TB free; free space before building.</p>"])
    raise RuntimeError(f"cdi_tm quota nearly full: {QTXT}")

PLAN = plan_months()
mt = pd.DataFrame([{"month": m, **{f"{s} dates": sum(ym_of(d) == m for d in SRC_DATES[s]) for s in SOURCES},
                    "status": "to build" if m in PLAN else "done"} for m in MONTHS])
ov = pd.DataFrame([{"data_source": s, "first": SRC_DATES[s][0], "last": SRC_DATES[s][-1], "dates": len(SRC_DATES[s])}
                   for s in SOURCES])
show(["<h3>Inputs and month plan</h3>",
      tbl(pd.DataFrame([("Months", f"{MONTHS[0]} to {MONTHS[-1]} ({len(MONTHS)})"),
                        ("To build", f"{len(PLAN)}" + (f" (from {PLAN[0]})" if PLAN else "")),
                        ("PKG power ids in scope", f"{sc['n']:,} ({sc['org']:,} organisations)"),
                        ("Output folders", f"{PATH_M}" + (f" | {PATH_D}" if WRITE_DAILY else "")),
                        ("Quota", QTXT),
                        ("Rough need", f"~0.3 GB month table + ~1 GB daily table per month, x {HDFS_REPLICATION} copies "
                                       f"= ~{len(PLAN) * 1.3 * HDFS_REPLICATION / 1000:.2f} TB for {len(PLAN)} months (measured after the first month)"),
                        ("Signature", SIG)], columns=["item", "value"])),
      "<h4>Sources</h4>", tbl(ov), "<h4>Business days per month and source</h4>", tbl(mt)])


In [ ]:
# ============================================================
# 3. Checks on the last month-end (one day per source)
# ============================================================
FAM = F.upper(F.trim(F.col("deposit_family").cast("string")))
BUCKET = (F.when(FAM.contains("SWEEP") & FAM.contains("OFF"), "sweep_off")
           .when(FAM.contains("SWEEP"), "sweep_on")
           .when(FAM.isin("NIBDDA", "IBDDA"), "dda")
           .when(FAM == "RETAIL", "retail")
           .when(FAM == "MMDA", "mmda")
           .when(FAM == "CD", "cd")
           .otherwise("other"))
BUCKETS = ["dda", "retail", "mmda", "cd", "sweep_on", "sweep_off", "other"]

mL = MONTHS[-1]
eom = {s: max(d for d in SRC_DATES[s] if ym_of(d) == mL) for s in SOURCES}
cond = None
for s, d in eom.items():
    c = (F.col("data_source") == s) & (F.col("src_load_date") == d)
    cond = c if cond is None else (cond | c)
x = (DM.filter(cond)
       .select("data_source", F.trim(F.col("acct_full_acct_id")).alias("acct"),
               F.coalesce(F.col("certificate_number").cast("string"), F.lit("")).alias("cert"),
               clean_pwr("cust_pwr_id").alias("pwr"), F.col("deposit_family").alias("family"), BUCKET.alias("bucket"),
               F.col("balance").cast("double").alias("bal"), F.upper(F.trim(F.col("acct_status_desc"))).alias("status"))
       .join(SCOPE.withColumnRenamed("customer_pwr_id", "pwr").withColumn("in_scope", F.lit(1)), "pwr", "left")
       .persist(StorageLevel.MEMORY_AND_DISK))
tot = x.agg(F.count(F.lit(1)).alias("n"), F.sum("bal").alias("bal")).collect()[0]
fam = (x.groupBy("data_source", "family", "bucket")
         .agg(F.count(F.lit(1)).alias("rows"), F.sum("bal").alias("bal")).orderBy(F.col("bal").desc()).toPandas())
fam["rows"] = fam["rows"].map(lambda v: f"{int(v):,}")
fam["balance share"] = fam["bal"].map(lambda v: fpct((v or 0) / tot["bal"]) if tot["bal"] else "")
fam["bal"] = fam["bal"].map(fnum)
stt = x.groupBy("status").agg(F.count(F.lit(1)).alias("rows"), F.sum("bal").alias("bal")).orderBy(F.col("rows").desc()).toPandas()
stt["rows"] = stt["rows"].map(lambda v: f"{int(v):,}")
stt["bal"] = stt["bal"].map(fnum)
cov = x.agg(F.avg(F.col("pwr").isNotNull().cast("int")).alias("pwr_rows"),
            F.sum(F.when(F.col("pwr").isNotNull(), F.col("bal"))).alias("pwr_bal"),
            F.avg(F.coalesce(F.col("in_scope"), F.lit(0))).alias("scope_rows"),
            F.sum(F.when(F.col("in_scope") == 1, F.col("bal"))).alias("scope_bal"),
            F.sum(F.when(F.col("is_org") == 1, F.col("bal"))).alias("org_bal")).collect()[0]
dup = (x.groupBy("data_source", "acct", "cert").count().filter("count > 1")
         .agg(F.count(F.lit(1)).alias("k"), F.sum("count").alias("r")).collect()[0])
multi = x.groupBy("data_source", "acct").agg(F.countDistinct("cert").alias("c")).filter("c > 1").count()
cross = (x.select("data_source", "acct").distinct().groupBy("acct").agg(F.countDistinct("data_source").alias("k"))
          .filter("k > 1").count())
x.unpersist()
COV = pd.DataFrame([
    ("Month-end dates used", "; ".join(f"{s} {d}" for s, d in eom.items())),
    ("Accounts (rows) / balance", f"{tot['n']:,} / {fnum(tot['bal'])}"),
    ("Rows with a power id / their balance", f"{fpct(cov['pwr_rows'])} / {fpct((cov['pwr_bal'] or 0) / tot['bal']) if tot['bal'] else ''}"),
    ("Rows of PKG customers / their balance", f"{fpct(cov['scope_rows'])} / {fpct((cov['scope_bal'] or 0) / tot['bal']) if tot['bal'] else ''}"),
    ("Balance of PKG organisations", fpct((cov["org_bal"] or 0) / tot["bal"]) if tot["bal"] else ""),
    ("Exact repeats (same source, account, certificate)", f"{(dup['k'] or 0):,} keys on {(dup['r'] or 0):,} rows (latest load kept)"),
    ("Accounts with 2+ certificates", f"{multi:,} (summed)"),
    ("Account numbers in 2+ sources", f"{cross:,} (kept apart: account = source + number)"),
], columns=["check", "value"])
show([f"<h3>Checks — month-end of {esc(mL)}</h3>", tbl(COV),
      "<h4>Deposit family by source and bucket (largest balance first)</h4>", tbl(fam),
      "<h4>Account status</h4>", tbl(stt)])


In [ ]:
# ============================================================
# 4. Build month by month
# ============================================================
LOG_CSV = os.path.join(OUT_DIR, "dep_cust_runlog.csv")
CLOSED, DORMANT = ["CLOSED"], ["DORMANT", "INACTIVE"]

def close_enough(a, b):
    a, b = float(a or 0), float(b or 0)
    return abs(a - b) <= max(1e-6 * max(abs(a), abs(b)), 0.01)

def lit_df(rows, names):
    """Small DataFrame from Python tuples without pandas / pickling (array of structs, exploded)."""
    arr = F.array(*[F.struct(*[F.lit(v).alias(n) for v, n in zip(r, names)]) for r in rows])
    return spark.range(1).select(F.explode(arr).alias("_r")).select("_r.*")

def raw(pairs):
    """Account rows for {source: [dates]}, one row per source / date / account / certificate (latest load)."""
    cond = None
    for s, ds in pairs.items():
        if ds:
            c = (F.col("data_source") == s) & F.col("src_load_date").isin(list(ds))
            cond = c if cond is None else (cond | c)
    w = Window.partitionBy("data_source", "src_load_date", "acct", "cert").orderBy(F.col("_ld").desc_nulls_last())
    return (DM.filter(cond)
              .select("data_source", "src_load_date", F.trim(F.col("acct_full_acct_id")).alias("acct"),
                      F.coalesce(F.col("certificate_number").cast("string"), F.lit("")).alias("cert"),
                      clean_pwr("cust_pwr_id").alias("pwr"), BUCKET.alias("bucket"),
                      F.col("balance").cast("double").alias("bal"),
                      F.upper(F.trim(F.col("acct_status_desc"))).alias("status"),
                      F.col("earnings_credit_rate").cast("double").alias("ecr"),
                      F.col("curr_int_rate").cast("double").alias("irate"),
                      F.col("interest_paid_mtd").cast("double").alias("ipaid"),
                      F.col("cod_hdfs_load_ts").cast("string").alias("_ld"))
              .filter(F.col("acct").isNotNull() & (F.col("acct") != ""))
              .withColumn("_r", F.row_number().over(w)).filter("_r = 1").drop("_r", "_ld"))

def good_dates(months):
    """Per source: load dates in the months, without partial loads (< LOW_LOAD_SHARE of the source's monthly median)."""
    ds = {s: [d for d in SRC_DATES[s] if ym_of(d) in months] for s in SOURCES}
    cond = None
    for s, dl in ds.items():
        c = (F.col("data_source") == s) & F.col("src_load_date").isin(dl)
        cond = c if cond is None else (cond | c)
    cnt = DM.filter(cond).groupBy("data_source", "src_load_date").count().toPandas()
    cnt["m"] = cnt["src_load_date"].str[:7]
    cnt["med"] = cnt.groupby(["data_source", "m"])["count"].transform("median")
    bad = cnt[cnt["count"] < LOW_LOAD_SHARE * cnt["med"]]
    keep = {s: sorted(set(ds[s]) - set(bad.loc[bad["data_source"] == s, "src_load_date"])) for s in SOURCES}
    for s in SOURCES:
        for mm in months:
            n_all = sum(ym_of(d) == mm for d in ds[s])
            n_ok = sum(ym_of(d) == mm for d in keep[s])
            if n_all and n_ok < 0.7 * n_all:
                raise RuntimeError(f"{mm}: {n_all - n_ok} of {n_all} {s} loads are partial; check the source")
    return keep, bad

def build_month(m):
    t0 = time.time()
    mp = ym_add(m, -1)
    keep, bad = good_dates([mp, m])
    days = sorted(set(d for s in SOURCES for d in keep[s] if ym_of(d) == m))
    if not days:
        raise RuntimeError(f"{m}: no business days")
    # calendar alignment: each source's balance for every business day (carry forward up to CARRY_DAYS)
    rows, carried = [], {s: 0 for s in SOURCES}
    for s in SOURCES:
        sd = pd.to_datetime(pd.Series(keep[s]))
        for d in days:
            prior = sd[sd <= pd.Timestamp(d)]
            if prior.empty or (pd.Timestamp(d) - prior.max()).days > CARRY_DAYS:
                raise RuntimeError(f"{m}: {s} has no load within {CARRY_DAYS} days before {d}")
            use = prior.max().strftime("%Y-%m-%d")
            carried[s] += int(use != d)
            rows.append((s, use, d, int(use != d)))
    MAP = lit_df(rows, ["data_source", "src_load_date", "bal_date", "carried"])
    need = {s: sorted(set(r[1] for r in rows if r[0] == s)) for s in SOURCES}

    r = raw(need).persist(StorageLevel.MEMORY_AND_DISK)
    ad = r.join(F.broadcast(MAP), ["data_source", "src_load_date"]).drop("src_load_date")
    ad = ad.join(SCOPE.withColumnRenamed("customer_pwr_id", "pwr"), "pwr", "left").persist(StorageLevel.MEMORY_AND_DISK)
    a0 = ad.agg(F.count(F.lit(1)).alias("rows"), F.sum("bal").alias("bal"),
                F.sum(F.when(F.col("is_org").isNotNull(), F.col("bal"))).alias("scope_bal"),
                F.sum(F.when(F.col("pwr").isNotNull(), F.col("bal"))).alias("pwr_bal")).collect()[0]
    ins = ad.filter(F.col("is_org").isNotNull())

    # ---- customer x day ----
    acct_id = F.concat_ws("|", "data_source", "acct")
    cd = (ins.groupBy(F.col("pwr").alias("customer_pwr_id"), "is_org", "bal_date")
             .agg(F.sum(F.when(F.col("bucket") != "sweep_off", F.col("bal")).otherwise(0.0)).alias("dep_bal"),
                  *[F.sum(F.when(F.col("bucket") == b, F.col("bal")).otherwise(0.0)).alias(f"bal_{b}") for b in BUCKETS],
                  F.countDistinct(acct_id).alias("n_acct"),
                  F.countDistinct(F.when(F.col("bal") < 0, acct_id)).alias("n_acct_negative"),
                  F.max("carried").alias("carried"))
             .persist(StorageLevel.MEMORY_AND_DISK))
    rc = cd.agg(*[F.sum(f"bal_{b}").alias(b) for b in BUCKETS], F.count(F.lit(1)).alias("n")).collect()[0]
    if not close_enough(sum(rc[b] or 0 for b in BUCKETS), a0["scope_bal"]):
        raise RuntimeError(f"{m}: customer-day balances do not add up to account balances")

    # ---- grid: every customer with deposits this month x every business day (absent = 0) ----
    DAYS = lit_df([(d,) for d in days], ["bal_date"])
    grid = (cd.select("customer_pwr_id", "is_org").distinct().crossJoin(F.broadcast(DAYS))
              .join(cd.drop("is_org"), ["customer_pwr_id", "bal_date"], "left")
              .withColumn("present", F.col("n_acct").isNotNull().cast("int"))
              .fillna(0.0, subset=["dep_bal"] + [f"bal_{b}" for b in BUCKETS])
              .fillna(0, subset=["n_acct", "n_acct_negative"]))
    wr = Window.partitionBy("customer_pwr_id").orderBy("bal_date").rowsBetween(Window.unboundedPreceding, 0)
    grid = (grid.withColumn("_peak", F.max("dep_bal").over(wr))
                .withColumn("_dd", F.col("_peak") - F.col("dep_bal"))
                .withColumn("_ddp", F.when(F.col("_peak") > 0, F.col("_dd") / F.col("_peak"))))
    mon = (grid.groupBy("customer_pwr_id", "is_org")
               .agg(F.count(F.lit(1)).alias("n_days"), F.sum("present").alias("n_days_present"),
                    F.expr("min_by(dep_bal, bal_date)").alias("dep_bal_first"),
                    F.expr("max_by(dep_bal, bal_date)").alias("dep_bal_eom"),
                    F.avg("dep_bal").alias("dep_bal_mean"), F.min("dep_bal").alias("dep_bal_min"),
                    F.max("dep_bal").alias("dep_bal_max"), F.stddev_pop("dep_bal").alias("dep_bal_std"),
                    F.max("_dd").alias("dep_drawdown_max"), F.max("_ddp").alias("dep_drawdown_pct"),
                    F.sum((F.col("dep_bal") < 0).cast("int")).alias("days_negative"),
                    F.greatest(-F.min("dep_bal"), F.lit(0.0)).alias("overdraft_max"),
                    *[x for b in BUCKETS for x in (F.expr(f"max_by(bal_{b}, bal_date)").alias(f"bal_{b}_eom"),
                                                   F.avg(f"bal_{b}").alias(f"bal_{b}_mean"))],
                    F.sum("n_acct_negative").alias("acct_days_negative")))

    # ---- account events against the previous month-end ----
    last = days[-1]
    def acct_level(df):
        return (df.groupBy("data_source", "acct")
                  .agg(F.max("pwr").alias("pwr"), F.min(F.col("status").isin(CLOSED).cast("int")).alias("closed"),
                       F.min(F.col("status").isin(DORMANT).cast("int")).alias("dormant")))
    cur_eom = acct_level(ad.filter(F.col("bal_date") == last))
    prev_dates = {s: [max(d for d in keep[s] if ym_of(d) == mp)] for s in SOURCES}
    prv = acct_level(raw(prev_dates)).persist(StorageLevel.MEMORY_AND_DISK)
    seen = ad.groupBy("data_source", "acct").agg(F.max("pwr").alias("pwr"))
    p = prv.select("data_source", "acct", F.col("closed").alias("p_closed"), F.col("dormant").alias("p_dormant"),
                   F.lit(1).alias("in_prev"))
    ev = (seen.join(cur_eom.drop("pwr"), ["data_source", "acct"], "left").join(p, ["data_source", "acct"], "left")
              .groupBy(F.col("pwr").alias("customer_pwr_id"))
              .agg(F.countDistinct(F.concat_ws("|", "data_source", "acct")).alias("n_acct"),
                   F.sum(F.when(F.col("closed") == 0, 1).otherwise(0)).alias("n_acct_eom_open"),
                   F.sum(F.col("in_prev").isNull().cast("int")).alias("n_acct_new"),
                   F.sum(((F.col("closed") == 1) & (F.coalesce(F.col("p_closed"), F.lit(0)) == 0)).cast("int")).alias("n_acct_closed"),
                   F.sum(((F.col("dormant") == 1) & (F.coalesce(F.col("p_dormant"), F.lit(0)) == 0)).cast("int")).alias("n_acct_dormant_new")))
    gone = (prv.join(seen.select("data_source", "acct").withColumn("_s", F.lit(1)), ["data_source", "acct"], "left")
               .filter(F.col("_s").isNull() & F.col("pwr").isNotNull())
               .groupBy(F.col("pwr").alias("customer_pwr_id")).agg(F.count(F.lit(1)).alias("n_acct_gone")))
    pos = F.when(F.col("bal") > 0, F.col("bal"))
    rates = (ins.filter(F.col("bal_date") == last).groupBy(F.col("pwr").alias("customer_pwr_id"))
                .agg((F.sum(F.when(F.col("ecr") > 0, F.col("ecr") * pos)) / F.sum(F.when(F.col("ecr") > 0, pos))).alias("ecr_wavg"),
                     (F.sum(F.when((F.col("irate") > 0) & (F.col("bucket") != "sweep_off"), F.col("irate") * pos))
                      / F.sum(F.when((F.col("irate") > 0) & (F.col("bucket") != "sweep_off"), pos))).alias("int_rate_wavg"),
                     F.sum("ipaid").alias("interest_paid_month")))

    out = (mon.join(ev, "customer_pwr_id", "left").join(rates, "customer_pwr_id", "left")
              .join(gone, "customer_pwr_id", "full").drop("is_org")
              .join(SCOPE, "customer_pwr_id", "inner")
              .fillna(0, subset=["n_days", "n_days_present", "n_acct", "n_acct_eom_open", "n_acct_new", "n_acct_closed",
                                 "n_acct_dormant_new", "n_acct_gone", "days_negative", "acct_days_negative"])
              .withColumn("has_dep", (F.col("n_days_present") > 0).cast("int")))
    cols = (["customer_pwr_id", "is_org", "has_dep", "n_days", "n_days_present", "dep_bal_first", "dep_bal_eom",
             "dep_bal_mean", "dep_bal_min", "dep_bal_max", "dep_bal_std", "dep_drawdown_max", "dep_drawdown_pct",
             "days_negative", "overdraft_max"] + [f"bal_{b}_{k}" for b in BUCKETS for k in ("eom", "mean")]
            + ["n_acct", "n_acct_eom_open", "n_acct_new", "n_acct_closed", "n_acct_dormant_new", "n_acct_gone",
               "acct_days_negative", "ecr_wavg", "int_rate_wavg", "interest_paid_month"])
    out = out.select(*cols).persist(StorageLevel.MEMORY_AND_DISK)

    ro = out.agg(F.count(F.lit(1)).alias("rows"), F.countDistinct("customer_pwr_id").alias("ids"),
                 F.sum("has_dep").alias("with_dep"), F.sum(F.col("is_org") * F.col("has_dep")).alias("orgs"),
                 F.sum("dep_bal_eom").alias("dep_eom"), F.sum("bal_sweep_off_eom").alias("off_eom"),
                 F.sum("n_acct_new").alias("new"), F.sum("n_acct_closed").alias("closed"),
                 F.sum("n_acct_gone").alias("gone"), F.sum("n_acct_dormant_new").alias("dormant")).collect()[0]
    eom_in = ins.filter(F.col("bal_date") == last).agg(F.sum("bal")).collect()[0][0]
    g_src = gone.join(SCOPE, "customer_pwr_id").agg(F.sum("n_acct_gone")).collect()[0][0] or 0
    bad_chk = [x for x, ok in (("one row per power id", ro["rows"] == ro["ids"]),
                               ("month-end balance", close_enough((ro["dep_eom"] or 0) + (ro["off_eom"] or 0), eom_in)),
                               ("accounts gone", (ro["gone"] or 0) == g_src)) if not ok]
    if bad_chk:
        raise RuntimeError(f"{m}: monthly table does not reconcile: {bad_chk}")

    # ---- write, then markers ----
    for p_ in (PATH_M, PATH_D):
        fs_delete(f"{p_}/month={m}/_PKG_MARKER.json")
    out.repartition(16).write.mode("overwrite").parquet(f"{PATH_M}/month={m}")
    n_daily = 0
    if WRITE_DAILY:
        dly = (cd.filter(F.col("is_org") == 1)
                 .select("customer_pwr_id", F.to_date("bal_date").alias("bal_date"), "dep_bal",
                         *[f"bal_{b}" for b in BUCKETS], "n_acct", "n_acct_negative", "carried"))
        dly.repartition(32, "customer_pwr_id").write.mode("overwrite").parquet(f"{PATH_D}/month={m}")
        n_daily = spark.read.parquet(f"{PATH_D}/month={m}").count()
        n_exp = cd.filter(F.col("is_org") == 1).count()
        if n_daily != n_exp:
            raise RuntimeError(f"{m}: daily table rows {n_daily} vs {n_exp}")
    n_rep = set_replication(f"{PATH_M}/month={m}", HDFS_REPLICATION)
    if WRITE_DAILY:
        n_rep += set_replication(f"{PATH_D}/month={m}", HDFS_REPLICATION)
    used_m = space_used(f"{PATH_M}/month={m}") + (space_used(f"{PATH_D}/month={m}") if WRITE_DAILY else 0)
    mk = json.dumps({"sig": SIG, "build_id": uuid.uuid4().hex[:12], "month": m,
                     "built": _dt.datetime.now().isoformat(timespec="seconds")})
    fs_write(f"{PATH_M}/month={m}/_PKG_MARKER.json", mk)
    if WRITE_DAILY:
        fs_write(f"{PATH_D}/month={m}/_PKG_MARKER.json", mk)
    for d_ in (r, ad, cd, prv, out):
        d_.unpersist()
    return {"month": m, "business days": len(days),
            "carried days": "; ".join(f"{s} {carried[s]}" for s in SOURCES if carried[s]),
            "partial loads dropped": "; ".join(f"{a} {b}" for a, b in zip(bad["data_source"], bad["src_load_date"])
                                               if ym_of(b) == m),
            "account-day rows": a0["rows"], "balance with power id": (a0["pwr_bal"] or 0) / a0["bal"] if a0["bal"] else None,
            "balance of PKG customers": (a0["scope_bal"] or 0) / a0["bal"] if a0["bal"] else None,
            "customers": ro["rows"], "with deposits": ro["with_dep"], "organisations": ro["orgs"],
            "deposits month-end": ro["dep_eom"], "sweep off-BS month-end": ro["off_eom"],
            "new": ro["new"], "closed": ro["closed"], "gone": ro["gone"], "turned dormant": ro["dormant"],
            "daily rows": n_daily, "space (GB, with copies)": round(used_m / 1e9, 3),
            "files set to fewer copies": n_rep, "seconds": round(time.time() - t0)}

def log_view(rows):
    if not rows:
        return "<p class='note'>(nothing built yet)</p>"
    v = pd.DataFrame(rows)
    for c in ("balance with power id", "balance of PKG customers"):
        v[c] = v[c].map(fpct)
    for c in ("deposits month-end", "sweep off-BS month-end"):
        v[c] = v[c].map(fnum)
    for c in ("account-day rows", "customers", "with deposits", "organisations", "new", "closed", "gone",
              "turned dormant", "daily rows"):
        v[c] = v[c].map(lambda x: f"{int(x or 0):,}")
    return tbl(v)

PLAN = plan_months()
done_now = []
if not PLAN:
    show(["<h3>Build</h3><p>All months are up to date; nothing to build.</p>"])
else:
    hd = display(HTML(CSS + f"<div class='cx'><h3>Build</h3><p>{len(PLAN)} months to build ...</p></div>"), display_id=True)
    t_all = time.time()
    def qline(i):
        qf, qb, qu = quota(LOC)
        per = np.mean([r["space (GB, with copies)"] for r in done_now]) if done_now else None
        left = len(PLAN) - i
        txt = (f"Quota: {qu / TB:.2f} of {qb / TB:.2f} TB used, {(qb - qu) / TB:.2f} TB free." if qb is not None
               else "No space quota found.")
        if per is not None:
            need = per * left / 1000
            txt += f" Measured {per:.2f} GB per month; {left} months left need ~{need:.2f} TB."
            if qb is not None and (qb - qu) / TB - need < MIN_FREE_TB:
                txt += " WARNING: not enough room to finish with the margin kept."
        return qb, qu, f"<p class='{'warn' if 'WARNING' in txt else 'note'}'>{esc(txt)}</p>"
    try:
        for i, m in enumerate(PLAN):
            qb, qu, ql = qline(i)
            if qb is not None and qb - qu < MIN_FREE_TB * TB:
                raise RuntimeError(f"Stopped before {m}: cdi_tm quota has {(qb - qu) / TB:.2f} TB free "
                                   f"(< {MIN_FREE_TB} TB). Free space, then rerun this cell; finished months are kept.")
            done_now.append(build_month(m))
            hd.update(HTML(CSS + f"<div class='cx'><h3>Build — {i + 1} of {len(PLAN)} months done, "
                                 f"{(time.time() - t_all) / 60:.0f} min so far</h3>" + qline(i + 1)[2]
                           + log_view(done_now) + "</div>"))
    finally:
        if done_now:
            old = pd.read_csv(LOG_CSV, dtype={"month": str}) if os.path.exists(LOG_CSV) else pd.DataFrame()
            new = pd.DataFrame(done_now)
            if len(old):
                old = old[~old["month"].isin(new["month"])]
            pd.concat([old, new]).sort_values("month").to_csv(LOG_CSV, index=False)
    hd.update(HTML(CSS + f"<div class='cx'><h3>Build — {len(PLAN)} months built in {(time.time() - t_all) / 60:.0f} min</h3>"
                   + log_view(done_now) + f"<p class='note'>Run log: {esc(os.path.abspath(LOG_CSV))}</p></div>"))


In [ ]:
# ============================================================
# 5. Register tables, monthly summary, column dictionary
# ============================================================
READY = [m for m in MONTHS if m not in plan_months(use_rebuild=False)]
if not READY:
    raise RuntimeError("No month is complete; run cell 4 first")

def register(table, path, months):
    full = f"{DB}.{table}"
    sch = spark.read.parquet(f"{path}/month={months[-1]}").schema
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
    if spark.sql(f"SHOW TABLES IN {DB} LIKE '{table}'").count() > 0:
        d = spark.sql(f"DESCRIBE TABLE EXTENDED {full}").toPandas()
        loc = d.loc[d["col_name"].astype(str).str.strip() == "Location", "data_type"].astype(str).tolist()
        if not loc or loc[0].rstrip("/") != path.rstrip("/"):
            raise RuntimeError(f"{full} already exists at {loc}; not replacing it")
        spark.sql(f"DROP TABLE {full}")          # external table: files stay
    spark.sql(f"CREATE TABLE {full} ({cols}, month STRING) USING parquet PARTITIONED BY (month) LOCATION '{path}'")
    for m in months:
        spark.sql(f"ALTER TABLE {full} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{path}/month={m}'")
    return full

fm = register(T_MONTH, PATH_M, READY)
fd = register(T_DAILY, PATH_D, READY) if WRITE_DAILY else None
cm = spark.table(fm).groupBy("month").count().toPandas().set_index("month")["count"]
LOG = pd.read_csv(LOG_CSV, dtype={"month": str}) if os.path.exists(LOG_CSV) else pd.DataFrame(columns=["month"])
LOG = LOG[LOG["month"].isin(READY)].set_index("month")
SUM = pd.DataFrame({"month": READY})
SUM["rows (table)"] = SUM["month"].map(lambda m: f"{int(cm.get(m, 0)):,}")
bad = []
if len(LOG):
    for c, f in (("business days", lambda x: f"{int(x)}"), ("carried days", str), ("partial loads dropped", str),
                 ("with deposits", lambda x: f"{int(x):,}"), ("organisations", lambda x: f"{int(x):,}"),
                 ("balance of PKG customers", fpct), ("deposits month-end", fnum), ("sweep off-BS month-end", fnum),
                 ("new", lambda x: f"{int(x):,}"), ("closed", lambda x: f"{int(x):,}"), ("gone", lambda x: f"{int(x):,}"),
                 ("turned dormant", lambda x: f"{int(x):,}"), ("daily rows", lambda x: f"{int(x):,}")):
        SUM[c] = SUM["month"].map(lambda m: f(LOG.loc[m, c]) if m in LOG.index and pd.notna(LOG.loc[m, c]) else "")
    bad = [m for m in READY if m in LOG.index and int(cm.get(m, -1)) != int(LOG.loc[m, "customers"])]

B = "dda (NIBDDA, IBDDA), retail, mmda, cd, sweep_on, sweep_off, other"
DICT = pd.DataFrame([
    ("both", "month", "Calendar month of the balances (YYYY-MM, partition)"),
    ("both", "customer_pwr_id", "Power id, 10 digits (PKG customers from pkg_cust_dim)"),
    (T_MONTH, "is_org", "1 = any of the power id's mdm_ids is party_type O in pkg_cust_dim"),
    (T_MONTH, "has_dep", "1 = at least one deposit account on some business day of the month (0 = only accounts gone since last month)"),
    (T_MONTH, "n_days, n_days_present", "Business days in the month (union of source load dates) / days with an account"),
    (T_MONTH, "dep_bal_*", "Daily deposit balance (all families except sweep_off): first day, month-end, mean, min, max, std over every business day (no account = 0)"),
    (T_MONTH, "dep_drawdown_max / _pct", "Largest drop from the month's running peak, in dollars / as share of that peak"),
    (T_MONTH, "days_negative, overdraft_max", "Days with a negative total deposit balance; largest negative total (0 if none)"),
    (T_MONTH, "bal_<family>_eom / _mean", f"Balance by family, month-end and mean; families: {B}. sweep_off = investment funds, not deposits"),
    (T_MONTH, "n_acct, n_acct_eom_open", "Accounts (source + account number) seen in the month / not CLOSED at month-end"),
    (T_MONTH, "n_acct_new", "Accounts not present at the previous month-end"),
    (T_MONTH, "n_acct_closed", "Status CLOSED at month-end and not at the previous month-end"),
    (T_MONTH, "n_acct_dormant_new", "Status DORMANT or INACTIVE at month-end and not at the previous month-end"),
    (T_MONTH, "n_acct_gone", "Present at the previous month-end, absent all month (counted under the previous power id)"),
    (T_MONTH, "acct_days_negative", "Account-days with a negative balance"),
    (T_MONTH, "ecr_wavg, int_rate_wavg", "Earnings credit rate / interest rate at month-end, weighted by positive balance over accounts with a rate"),
    (T_MONTH, "interest_paid_month", "Sum of interest_paid_mtd at month-end"),
    (T_DAILY, "bal_date", "Business day (organisations only; rows only on days with an account)"),
    (T_DAILY, "dep_bal, bal_<family>", "Deposit balance and balance by family that day"),
    (T_DAILY, "n_acct, n_acct_negative", "Accounts / accounts with a negative balance that day"),
    (T_DAILY, "carried", "1 = at least one source had no (full) load that day and its previous balance was carried forward"),
], columns=["table", "column", "definition"])
DICT.to_csv(os.path.join(OUT_DIR, "dep_cust_dictionary.csv"), index=False)
show([f"<h3>Registered {esc(fm)}" + (f" and {esc(fd)}" if fd else "") + f" — {len(READY)} months, {READY[0]} to {READY[-1]}</h3>"]
     + ([f"<p class='warn'>Row counts differ from the run log for: {esc(', '.join(bad))}</p>"] if bad else [])
     + [tbl(SUM), "<p class='note'>In Impala: INVALIDATE METADATA " + esc(fm) + (f"; INVALIDATE METADATA {esc(fd)}" if fd else "") + ";</p>",
        "<h4>Columns</h4>", tbl(DICT)])
